# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset—clinicopathological and molecular data for second primary colorectal cancer in survivors—using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is defined by a Croissant schema, accessible via the following URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Let's review the available record sets and their fields, referencing entities by their `@id` fields as required by the Croissant specification.

In [ ]:
# Iterate through all record sets in the dataset and print their structure by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s) in the dataset.")

for rs in record_sets:
    print(f"Record Set: {rs.id}")  # @id of the record set
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
    print()

## 3. Data Extraction
Let's load data from a record set into a Pandas DataFrame for analysis. All operations will use record set and field `@id`s as identifiers.

In [ ]:
# For this dataset, there is likely one primary record set.
# We'll extract all records for each record set by @id.
dataframes = {}

for rs in record_sets:
    # Use the record set @id
    record_set_id = rs.id
    # Fetch all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for Record Set @id: {record_set_id}, Shape: {df.shape}")
        print(f"Columns (Field @id): {df.columns.tolist()}")
        print(df.head(3).to_string())
    else:
        print(f"No records returned for Record Set @id: {record_set_id}")


## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing steps, using only `@id` to reference all columns (fields). As an example, we'll:
- Select a numeric field (e.g., patient's age or a quantitative measurement field, as available)
- Filter records based on a threshold
- Normalize a field
- Group or aggregate by a key variable

Refer to the record set and field `@id`s printed above for the actual available fields.

In [ ]:
# Replace this with the correct record set @id (usually just one in this dataset)
record_set_id = None
if len(record_sets) > 0:
    record_set_id = record_sets[0].id

df = dataframes[record_set_id]

# Identify a numeric field. We'll try to pick one with 'Age' or similar, else just the first numeric one.
numeric_field_id = None
for field in record_sets[0].fields:
    if 'age' in field.name.lower() or field.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']:
        numeric_field_id = field.id
        break

print(f"Selected Numeric Field (by @id): {numeric_field_id}")

if numeric_field_id and numeric_field_id in df.columns:
    # Choose a threshold appropriate for age; otherwise, use 50 as a default
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold]
    display_cols = [numeric_field_id]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[display_cols].head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (z-score):")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Find a categorical/grouping field (e.g., sex/gender, tumor site, etc.)
    group_field_id = None
    for field in record_sets[0].fields:
        # Skip numeric field
        if field.id == numeric_field_id:
            continue
        if field.data_type == 'schema:Text' or 'site' in field.name.lower() or 'sex' in field.name.lower() or 'gender' in field.name.lower():
            group_field_id = field.id
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable grouping field found for aggregation.")
else:
    print("No suitable numeric field found in the record set for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and if available, show differences by group. For all plots, reference columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution histogram of the numeric field (referenced by @id)
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped, show group differences (e.g., boxplot by group_field_id)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
This notebook used `mlcroissant` to load and examine the FAIR² clinicopathological dataset for second primary colorectal cancer survivors. Data was explored by referencing entities with their `@id`, following FAIR data principles. We showed how to identify available variables, filter and normalize numeric fields, group and analyze by key strata, and visualize variable distributions. 

For further analysis, see the [FAIR² project page](https://sen.science/doi/10.71728/senscience.qs2f-h81p) or the dataset documentation for details on data semantics and suggested use cases.